# DQN: Online Training Directly on Hardware

This notebook trains the Q-network **on the physical unbalanced disk** instead of the simulator.
This eliminates the sim-to-real gap entirely — the agent learns from real hardware dynamics,
real friction, real sensor noise, and real actuator responses.

### Key differences from simulator training:
- **No simulator environment** — `UnbalancedDisk_exp` is used throughout
- **Online experience replay** — a rolling replay buffer is filled step-by-step on hardware
- **Smaller rollouts per iteration** — physical time is the bottleneck, not compute
- **Reward computed manually** — hardware env may not have the same reward API
- **Conservative epsilon schedule** — hardware safety requires more exploration early on
- **Graceful shutdown** — `finally` blocks ensure the disk is always stopped safely

## 1. Imports and Setup

In [ ]:
import gym_unbalanced_disk
import numpy as np
import torch
from torch import nn
from copy import deepcopy
from collections import deque
import random
import time
from matplotlib import pyplot as plt

## 2. Discrete Action Space

Same 17-action voltage grid as the simulator training.

In [ ]:
discrete_actions = np.array([
    -3.0, -2.0, -1.0, -0.5, -0.25,
    -0.1, -0.05, -0.02,
     0.0,
     0.02, 0.05, 0.1,
     0.25, 0.5, 1.0, 2.0, 3.0
])
num_actions = len(discrete_actions)
print(f'Action space: {num_actions} discrete voltages from {discrete_actions[0]}V to {discrete_actions[-1]}V')

## 3. Q-Network

Input is always `[sin(θ), cos(θ), ω]` (3 values), matching the simulator network architecture.
This also means **a checkpoint from the simulator can be loaded as a warm start** (see Section 7).

In [ ]:
OBS_DIM = 3  # [sin(theta), cos(theta), omega]

class Qfunction(nn.Module):
    def __init__(self):
        super(Qfunction, self).__init__()
        self.lay1 = nn.Linear(OBS_DIM, 128)
        self.F1   = nn.Tanh()
        self.lay2 = nn.Linear(128, num_actions)

    def forward(self, obs):
        return self.lay2(self.F1(self.lay1(obs)))

# Quick sanity check
Q_test = Qfunction()
dummy = torch.zeros(1, OBS_DIM)
print('Q output shape:', Q_test(dummy).shape)  # expect (1, 17)

## 4. Hardware Observation Helper

`UnbalancedDisk_exp` returns raw `[θ, ω]`.
This function converts to `[sin(θ), cos(θ), ω]` — the format the network expects.

In [ ]:
def hw_obs_to_sincos(raw_obs):
    """Convert hardware [theta, omega] -> network [sin(theta), cos(theta), omega]."""
    th    = float(raw_obs[0])
    omega = float(raw_obs[1])
    return np.array([np.sin(th), np.cos(th), omega], dtype=np.float32)


def hardware_reward(raw_obs):
    """
    Compute reward from raw hardware observation.
    Matches the simulator reward used during training:
        r = 0.5 * (1 - cos(theta))^2
    This peaks at 1 when theta=pi (top) and is 0 at bottom.

    Near the top (|cos(theta)| close to -1), switch to a balancing reward
    that also penalises angular velocity:
        r = 1.0 - 0.2 * omega^2
    """
    th    = float(raw_obs[0])
    omega = float(raw_obs[1])
    cos_th = np.cos(th)

    if cos_th < -0.95:  # within ~18 degrees of the top
        return float(1.0 - 0.2 * omega**2)
    else:
        return float(0.5 * (1.0 - cos_th)**2)


print('Reward at bottom (theta=0)  :', hardware_reward([0.0, 0.0]))
print('Reward at top    (theta=pi) :', hardware_reward([np.pi, 0.0]))
print('Reward at top, spinning fast :', hardware_reward([np.pi, 5.0]))

## 5. Replay Buffer

A standard fixed-size circular buffer. Experiences are added one step at a time
(online), then sampled in random mini-batches for training.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states),      dtype=torch.float32),
            torch.tensor(np.array(actions),     dtype=torch.long),
            torch.tensor(np.array(rewards),     dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(np.array(dones),       dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)

print('ReplayBuffer ready.')

## 6. Hardware Episode Runner

Collects `N_steps` real hardware steps into the replay buffer,
applying epsilon-greedy exploration.

**Episode logic:**
- If the hardware env signals `terminated` or `truncated`, a new episode begins.
- A settle delay after each reset gives the disk time to stabilise.
- Reward is computed locally via `hardware_reward()` (does not rely on env's reward).

In [ ]:
def collect_hardware_steps(Q, env, replay_buffer, epsilon, N_steps, settle_time=0.5):
    """
    Run N_steps on the physical hardware and push transitions into replay_buffer.

    Parameters
    ----------
    Q             : Qfunction  – current Q-network (used greedy, no grad)
    env           : UnbalancedDisk_exp instance
    replay_buffer : ReplayBuffer
    epsilon       : float  – exploration probability
    N_steps       : int    – number of hardware steps to collect
    settle_time   : float  – seconds to wait after a reset before resuming

    Returns
    -------
    episode_rewards : list of float – cumulative reward per completed episode
    """
    Qfun = lambda x: Q(torch.tensor(x[None, :], dtype=torch.float32))[0].detach().numpy()

    episode_rewards = []
    ep_reward = 0.0

    raw_obs, info = env.reset()
    time.sleep(settle_time)
    obs = hw_obs_to_sincos(raw_obs)

    with torch.no_grad():
        for step in range(N_steps):
            # --- epsilon-greedy action selection ---
            if np.random.uniform() > epsilon:
                action_idx = int(np.argmax(Qfun(obs)))
            else:
                action_idx = np.random.randint(0, num_actions)

            voltage = float(discrete_actions[action_idx])

            # --- hardware step ---
            raw_next, _env_reward, terminated, truncated, info = env.step(voltage)
            obs_next = hw_obs_to_sincos(raw_next)

            # Use our own reward (consistent with simulator training)
            reward = hardware_reward(raw_next)
            ep_reward += reward

            done = terminated or truncated
            replay_buffer.push(obs, action_idx, reward, obs_next, float(done))

            if done:
                episode_rewards.append(ep_reward)
                ep_reward = 0.0
                raw_obs, info = env.reset()
                time.sleep(settle_time)
                obs = hw_obs_to_sincos(raw_obs)
            else:
                obs = obs_next

    return episode_rewards

print('collect_hardware_steps() defined.')

## 7. (Optional) Warm-Start from Simulator Checkpoint

Loading a pre-trained simulator checkpoint dramatically reduces hardware time needed.
The network architecture is identical (`[sin θ, cos θ, ω]` → 128 → 17 actions),
so weights transfer directly.

**Skip this cell if you want to train from scratch on hardware.**

In [ ]:
WARM_START = True          # Set False to train from scratch
CHECKPOINT_PATH = 'Q-checkpoint'   # path to your simulator-trained checkpoint

Q = Qfunction()

if WARM_START:
    try:
        Q.load_state_dict(torch.load(CHECKPOINT_PATH))
        print(f'Loaded warm-start checkpoint from "{CHECKPOINT_PATH}"')
        print('Hardware fine-tuning will start from the simulator policy.')
    except FileNotFoundError:
        print(f'Warning: checkpoint "{CHECKPOINT_PATH}" not found — training from scratch.')
else:
    print('Training from scratch on hardware.')

## 8. Main Hardware DQN Training Loop

### Training schedule
| Phase | Iterations | Steps/iter | Purpose |
|-------|-----------|------------|---------|
| Warm-up | First `N_warmup_steps` total | — | Fill replay buffer before any gradient updates |
| Training | `N_iterations` | `N_steps_per_iter` | Collect + update interleaved |

### Safety notes
- The `finally` block **always** runs: hardware is stopped even on Ctrl+C.
- `settle_time` after each reset lets the disk stop spinning before the next episode.
- Keep an eye on the disk — if it behaves dangerously, use Ctrl+C.

In [ ]:
def DQN_hardware_train(
    Q,
    env,
    # --- Training schedule ---
    N_iterations          = 50,    # number of collect→update cycles
    N_steps_per_iter      = 500,   # hardware steps per iteration
    N_warmup_steps        = 2000,  # steps to collect before first gradient update
    # --- Learning ---
    gamma                 = 0.99,
    lr                    = 1e-3,
    batch_size            = 64,
    N_updates_per_iter    = 200,   # gradient steps per iteration
    target_update_freq    = 50,    # target net update every N gradient steps
    buffer_capacity       = 50_000,
    # --- Exploration ---
    epsilon_start         = 0.5,
    epsilon_end           = 0.05,
    # --- Hardware ---
    settle_time           = 0.5,   # seconds to wait after env.reset()
    checkpoint_path       = 'Q-checkpoint-hw',
):
    """
    Online DQN training loop running entirely on physical hardware.
    """
    optimizer    = torch.optim.Adam(Q.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(capacity=buffer_capacity)
    Q_target     = deepcopy(Q)
    Q_target.eval()

    best_score   = -float('inf')
    torch.save(Q.state_dict(), checkpoint_path)

    all_rewards  = []   # per-episode rewards across all iterations
    iter_scores  = []   # mean episode reward per iteration
    grad_step    = 0    # global gradient step counter

    # ------------------------------------------------------------------ #
    #  Phase 1: Warm-up — fill buffer with random/epsilon=1 transitions   #
    # ------------------------------------------------------------------ #
    print(f'=== Warm-up: collecting {N_warmup_steps} steps with epsilon=1.0 ===')
    try:
        ep_rewards = collect_hardware_steps(
            Q, env, replay_buffer,
            epsilon=1.0,
            N_steps=N_warmup_steps,
            settle_time=settle_time
        )
        print(f'Warm-up done. Buffer size: {len(replay_buffer)}')
    except KeyboardInterrupt:
        print('Warm-up interrupted. Stopping hardware.')
        env.reset()
        env.close()
        return

    # ------------------------------------------------------------------ #
    #  Phase 2: Training iterations                                        #
    # ------------------------------------------------------------------ #
    try:
        for iteration in range(N_iterations):
            # Linearly decay epsilon over all iterations
            frac    = iteration / max(N_iterations - 1, 1)
            epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)

            print(f'\n--- Iteration {iteration+1}/{N_iterations}  |  epsilon={epsilon:.3f}  |  buffer={len(replay_buffer)} ---')

            # --- 2a. Collect hardware experience ---
            ep_rewards = collect_hardware_steps(
                Q, env, replay_buffer,
                epsilon=epsilon,
                N_steps=N_steps_per_iter,
                settle_time=settle_time
            )
            all_rewards.extend(ep_rewards)

            if ep_rewards:
                print(f'  Episodes completed: {len(ep_rewards)}  |  Mean ep reward: {np.mean(ep_rewards):.3f}')
            else:
                print('  No complete episodes this iteration (episode still running).')

            # --- 2b. Gradient updates ---
            if len(replay_buffer) < batch_size:
                print('  Buffer too small to train yet — skipping gradient updates.')
                continue

            Q.train()
            losses = []

            for _ in range(N_updates_per_iter):
                states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

                # Target: r + gamma * max_a Q_target(s', a) * (1 - done)
                with torch.no_grad():
                    max_Q_next = Q_target(next_states).max(dim=1)[0]
                    td_target  = rewards + gamma * max_Q_next * (1.0 - dones)

                # Current Q(s, a) for the taken action
                Q_pred = Q(states).gather(1, actions.unsqueeze(1)).squeeze(1)

                loss = torch.mean((td_target - Q_pred) ** 2)

                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping for stability on hardware
                torch.nn.utils.clip_grad_norm_(Q.parameters(), max_norm=10.0)
                optimizer.step()

                losses.append(loss.item())
                grad_step += 1

                # Periodically copy Q → Q_target
                if grad_step % target_update_freq == 0:
                    Q_target.load_state_dict(Q.state_dict())

            Q.eval()
            print(f'  Mean loss this iteration: {np.mean(losses):.4f}')

            # --- 2c. Checkpoint best policy ---
            if ep_rewards:
                score = float(np.mean(ep_rewards))
                iter_scores.append(score)
                if score > best_score:
                    best_score = score
                    torch.save(Q.state_dict(), checkpoint_path)
                    print(f'  *** New best: {best_score:.3f} — checkpoint saved to "{checkpoint_path}" ***')
            else:
                iter_scores.append(np.nan)

    except KeyboardInterrupt:
        print('\nTraining interrupted by user.')

    finally:
        # Always stop the hardware safely
        print('\nStopping hardware safely...')
        try:
            env.reset()
            env.close()
        except Exception as e:
            print(f'  Warning during hardware shutdown: {e}')

        # Load best checkpoint
        try:
            Q.load_state_dict(torch.load(checkpoint_path))
            print(f'Loaded best checkpoint from "{checkpoint_path}" (score={best_score:.3f})')
        except Exception:
            print('Could not reload checkpoint.')

        print(f'\nTraining complete. Best hardware score: {best_score:.3f}')

    return all_rewards, iter_scores

print('DQN_hardware_train() defined.')

## 9. Run Hardware Training

### Recommended parameters for a first run

| Parameter | Value | Reasoning |
|-----------|-------|----------|
| `N_warmup_steps` | 2000 | ~50 seconds of random data to seed the buffer |
| `N_steps_per_iter` | 500 | ~12 seconds per iteration at `dt=0.025` |
| `N_iterations` | 50 | ~10 minutes total hardware time |
| `N_updates_per_iter` | 200 | Enough gradient steps without lag |
| `epsilon_start` | 0.5 | If warm-starting from sim checkpoint, less exploration needed |
| `epsilon_end` | 0.05 | Converge to mostly-greedy near the end |
| `settle_time` | 0.5 | Half-second for disk to settle after reset |

⚠️ **Monitor the physical disk throughout.** Press **Ctrl+C** at any time to stop safely.

In [ ]:
umax = 3.0

env = gym_unbalanced_disk.UnbalancedDisk_exp(
    umax=umax,
    dt=0.025
)

print('Hardware environment initialised.')
print(f'umax={umax}V, dt=0.025s')

In [ ]:
# If you skipped Section 7 and Q is not defined yet, create a fresh one:
# Q = Qfunction()

all_rewards, iter_scores = DQN_hardware_train(
    Q,
    env,
    N_iterations       = 50,
    N_steps_per_iter   = 500,
    N_warmup_steps     = 2000,
    gamma              = 0.99,
    lr                 = 5e-4,       # Lower than sim: hardware fine-tuning needs smaller steps
    batch_size         = 64,
    N_updates_per_iter = 200,
    target_update_freq = 50,
    buffer_capacity    = 50_000,
    epsilon_start      = 0.4,        # Lower if warm-starting, higher if training from scratch
    epsilon_end        = 0.05,
    settle_time        = 0.5,
    checkpoint_path    = 'Q-checkpoint-hw',
)

## 10. Plot Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if all_rewards:
    axes[0].plot(all_rewards, '.', alpha=0.5, markersize=3)
    # Rolling mean
    window = min(20, len(all_rewards))
    if len(all_rewards) >= window:
        rolling = np.convolve(all_rewards, np.ones(window)/window, mode='valid')
        axes[0].plot(range(window-1, len(all_rewards)), rolling, 'r-', linewidth=2, label=f'{window}-ep rolling mean')
        axes[0].legend()
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative reward')
axes[0].set_title('Hardware Training — Episode Rewards')

if iter_scores:
    valid = [(i, s) for i, s in enumerate(iter_scores) if not np.isnan(s)]
    if valid:
        xs, ys = zip(*valid)
        axes[1].plot(xs, ys, 'o-')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Mean episode reward')
axes[1].set_title('Mean Reward per Iteration')

plt.tight_layout()
plt.savefig('hardware_training_progress.png', dpi=150)
plt.show()
print('Plot saved.')

## 11. Evaluate the Hardware-Trained Policy

Runs the greedy policy on hardware and logs angle, velocity, and voltage at each step.

In [ ]:
Q.load_state_dict(torch.load('Q-checkpoint-hw'))
Q.eval()

# Re-open the hardware env if closed
env = gym_unbalanced_disk.UnbalancedDisk_exp(umax=3.0, dt=0.025)

theta_list   = []
omega_list   = []
voltage_list = []
reward_list  = []

print('=== HARDWARE EVALUATION (hardware-trained policy) ===')

try:
    raw_obs, info = env.reset()
    time.sleep(0.5)

    with torch.no_grad():
        Qfun = lambda x: Q(torch.tensor(x[None, :], dtype=torch.float32))[0].numpy()

        for step in range(600):
            obs_sincos = hw_obs_to_sincos(raw_obs)
            qvals      = Qfun(obs_sincos)
            action_idx = int(np.argmax(qvals))
            voltage    = float(discrete_actions[action_idx])

            theta_list.append(float(raw_obs[0]))
            omega_list.append(float(raw_obs[1]))
            voltage_list.append(voltage)
            reward_list.append(hardware_reward(raw_obs))

            if step % 20 == 0:
                theta_deg = np.degrees(raw_obs[0])
                print(f'step={step:4d} | theta={theta_deg:7.1f}° | omega={raw_obs[1]:8.3f} | u={voltage:5.2f}V')

            raw_obs, _, terminated, truncated, info = env.step(voltage)

            if terminated or truncated:
                print(f'Episode ended at step {step}')
                break

finally:
    print('Stopping hardware...')
    env.reset()
    env.close()

## 12. Evaluation Plots

In [ ]:
t_axis = np.arange(len(theta_list)) * 0.025  # seconds

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(t_axis, np.degrees(theta_list))
axes[0].axhline(180, color='r', linestyle='--', label='Target (180°)')
axes[0].axhline(-180, color='r', linestyle='--')
axes[0].set_ylabel('θ (degrees)')
axes[0].legend()
axes[0].set_title('Hardware Evaluation — Hardware-Trained Policy')

axes[1].plot(t_axis, omega_list)
axes[1].set_ylabel('ω (rad/s)')

axes[2].step(t_axis, voltage_list, where='post')
axes[2].set_ylabel('Voltage (V)')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig('hardware_eval.png', dpi=150)
plt.show()

print(f'Total reward: {sum(reward_list):.2f}')
print(f'Steps run:    {len(theta_list)}')